# étoiles de type Herbig
$\rightarrow$ **analyse de v1296Aql**
- 

## spectro dashboard
- lancer la cellule suivante
- sur 'Colormap', bouton droit : "create new view for cell output"
- redimensionner ou déplacer l'onglet créé 'Output View' 

In [ ]:
%matplotlib widget
import numpy as np
from spectro_dashboard import SpectroDashboard

# 1. Afficher le dashboard
db = SpectroDashboard()
db.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting
from specutils import Spectrum1D
from astropy import units as u


# chargement
filename = 'data/plouis/_v1295aql_20250822_847.fits'

# Lecture
sp = Spectrum1D.read(filename)
x_data = sp.spectral_axis.value
y_data = sp.flux.value 

# on affiche le tout
db.load_spectrum(x_data, y_data, label='Spectre Brut')


# ajustement et mesure de H beta

In [ ]:

# Extraction des valeurs brutes (pour éviter les soucis avec les unités astropy)
x_data = sp.spectral_axis.value
y_data = sp.flux.value 

# Découpage de la zone (4910 * u.AA, 4940 * u.AA)
mask = (x_data > 4840) & (x_data < 4880)
x_window = x_data[mask]
y_window = y_data[mask]

# Initialisation pour aider les modèles
y_max = np.max(y_window)
y_min = np.min(y_window)

# modele sur la raie de gauche
g_init_V = models.Gaussian1D(amplitude=y_max, mean=4855, stddev=2.0)

# modele sur la raie de droite
g_init_R = models.Gaussian1D(amplitude=y_max, mean=4868, stddev=2.0)

# on prépare le continuum 
continuum_init = models.Const1D(amplitude=y_min)

model_init = g_init_V + g_init_R + continuum_init

# On ajuste sur les données fenêtrées
fitter = fitting.LevMarLSQFitter()
fit = fitter(model_init, x_window, y_window)

# fit[0] est la Gaussienne V, fit[1] est la Gaussienne R
center_V = fit[0].mean.value
center_R = fit[1].mean.value

# Calcul de la vitesse
delta_lambda = abs(center_R - center_V)
lambda_0 = 4861 # H-beta repos
c = 299792.458
v_rot_disk = (c * (delta_lambda / lambda_0)) / 2

# on affiche le tout
db.load_spectrum(x_window, y_window, label='Spectre Brut')
db.load_spectrum(x_window, fit(x_window), label='Fit Double Gaussienne')

# Les Lignes Verticales (Centres des Gaussiennes)
l_v = db.ax_spec.axvline(center_V, color='blue', linestyle='-', alpha=0.8, label=f'Centre V ({center_V:.2f} A)')
l_r = db.ax_spec.axvline(center_R, color='green', linestyle='-', alpha=0.8, label=f'Centre R ({center_R:.2f} A)')

# et les résultats
print(f"Position du Pic Bleu (Modélisé)  : {center_V:.3f} A")
print(f"Position du Pic Rouge (Modélisé) : {center_R:.3f} A")
print(f"Séparation (Delta Lambda)        : {delta_lambda:.3f} A")
print("-" * 40)
print(f"VITESSE DE ROTATION DU DISQUE    : {v_rot_disk:.1f} km/s")
print("-" * 40)

## analyse :

<font size=4>

$433.0 \text{ km/s}$ représente la vitesse de rotation du gaz à l'endroit où la raie $\text{H}\beta$ est émise (son rayon de formation), projetée sur la ligne de visée (v sin(i))

- V1295 Aql est une étoile de Herbig Ae (type A) de masse intermédiaire (plusieurs fois la masse du Soleil).
- Sa forte gravité requiert des vitesses orbitales très élevées à proximité de l'étoile.
- Une vitesse de $433 \text{ km/s}$ est une forte indication que le $\text{H}\beta$ est formé dans la région la plus interne et la plus chaude du disque d'accrétion, là où le gaz est proche de la vitesse limite de rotation stable autour d'une étoile massive.

Calcul de la Vitesse Radiale du Centre ($V_r$) :

$$\lambda_{\text{centre}} = \frac{4854.913 \text{ A} + 4868.955 \text{ A}}{2} = \mathbf{4861.934 \text{ A}}$$

$$V_r (\text{H}\beta) = 299792 \text{ km/s} \cdot \left(\frac{4861.934 \text{ A}}{4861.32 \text{ A}} - 1\right) \approx \mathbf{+37.8 \text{ km/s}}$$

</font>


# ajustement et mesure de Fe I

In [ ]:
db.clear_spectra()

# Extraction des valeurs brutes (pour éviter les soucis avec les unités astropy)
x_data = sp.spectral_axis.value
y_data = sp.flux.value 

lambda_0 = 4923.93 # FeII repos
#lambda_0 = 5018.44 # FeII repos
#lambda_0 = 5169.03 # FeII repos

# Découpage de la zone (4910 * u.AA, 4940 * u.AA) $4923.
mask = (x_data > (lambda_0 - 10)) & ((x_data < lambda_0 + 10))
#mask = (x_data > 4917) & (x_data < 4934)
x_window = x_data[mask]
y_window = y_data[mask]

# Initialisation pour aider les modèles
y_max = np.max(y_window)
y_min = np.min(y_window)

# modele sur la raie 
g_init= models.Gaussian1D(amplitude=y_max, mean=lambda_0, stddev=2.0)

# on prépare le continuum 
continuum_init = models.Const1D(amplitude=y_min)

model_init = g_init + continuum_init

# On ajuste sur les données fenêtrées
fitter = fitting.LevMarLSQFitter()
fit = fitter(model_init, x_window, y_window)

# fit[0] est la Gaussienne V, fit[1] est la Gaussienne R
center_V = fit[0].mean.value

# Calcul de la vitesse
delta_lambda = abs(lambda_0 - center_V)

c = 299792.458
v_rot_disk = (c * (delta_lambda / lambda_0))

# on affiche le tout
db.load_spectrum(x_window, y_window, label='Spectre Brut')
db.load_spectrum(x_window, fit(x_window), label='Fit Gaussien')

# Les Lignes Verticales (Centres des Gaussiennes)
l_v = db.ax_spec.axvline(center_V, color='blue', linestyle='-', alpha=0.8, label=f'Centre V ({center_V:.2f} A)')

# et les résultats
print(f"Position du Pic (Modélisé)  : {center_V:.3f} A")
print(f"Delta Lambda                : {delta_lambda:.3f} A")
print("-" * 40)
print(f"VITESSE RADIALE             : {v_rot_disk:.1f} km/s")
print("-" * 40)


In [ ]:
from astropy.coordinates import SkyCoord, EarthLocation
from astropy.time import Time


# Coordonnées de la cible
coord = SkyCoord.from_name('V1296 Aql')

# Localisation du CALC
location = EarthLocation.from_geodetic(lon=-1.68*u.deg, lat=48.11*u.deg, height=50*u.m)

# Date d'observation depuis le header FITS
date_obs = Time(sp.meta['header']['DATE-OBS'], format="isot", scale="utc")
print(f"{date_obs=}")

# Correction barycentrique
corr = coord.radial_velocity_correction(obstime=date_obs, location=location).to(u.km/u.s)
print("Correction barycentrique :", corr)


# analyse

<font size="4">
    
L'étoile V1295 Aql (et son disque) se déplace dans la Galaxie à une vitesse de $V_{\text{abs}} = V_r + (-18.75 \text{ km/s})$ = $+19.05 \text{ km/s}$ : le système s'éloigne du Soleil

On a une structure gazeuse autour de cette étoile de Herbig Ae qui comprend :

- Rotation (H$\beta$ Ailes) : Rotation Képérienne extrêmement rapide du disque interne à $\mathbf{433.0 \text{ km/s}}$.

- Accrétion : Le gaz $\text{Fe II}$ à $5018 \text{ A}$ a une vitesse de $\mathbf{-12.15 \text{ km/s}}$ , inférieure à celle de l'étoile ($+19.05 \text{ km/s}$) -> flux de matière qui tombe sur l'étoile.

- Ejection/Vent : Le gaz $\text{Fe II}$ à $5169 \text{ A}$ a une vitesse de $\mathbf{+25.45 \text{ km/s}}$ , supérieure à la vitesse de l'étoile ($+19.05 \text{ km/s}$) -> il s'éloigne de l'étoile. -> Vent.

-> **Une jeune étoile en pleine construction**

</font>